数据说明：
- 下载地址：https://drive.google.com/file/d/1gaFy8RaQVUEXo2n0peCBR5gYKCB-mNHc/view
- 音频数据已经被预处理成了40维的梅尔频谱图(Mel-spectrogram)
- n_mels: 40 —— 音频特征的维度(固定为40维的梅尔频谱)。
- speakers —— 包含所有说话人信息的集合。
- "idXXXXX" —— 说话人的唯一ID，也就是模型需要预测的分类标签(Label)。
- 每个 ID 下的列表 —— 记录了该说话人名下的多段语音信息：
- feature_path：真实的语音特征张量文件(.pt文件)存放的路径。
- mel_len：这段语音的时间长度(帧数)，供读取数据时裁剪或填充使用。

In [1]:
import numpy as np
import torch
import torch.nn as nn
import json

from torch._C import dtype
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

# 库配置
# 库配置
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

定义数据集

In [ ]:
class SpeakerDataset(Dataset):
  def __init__(self, segment_len=128):
    self.segment_len = segment_len

    # 加载 speaker id 到整数的映射
    mapping = json.load(open("Dataset/mapping.json"))
    self.speaker2id = mapping["speaker2id"] #{"id10001": 0, "id10005": 1, "id10006": 2, ...}的形式

    # 加载音频路径
    metadata = json.load(open("Dataset/metadata.json"))["speakers"]
    """
    数据的形式：
    {
    "id10473": [
      {
        "feature_path": "uttr-5c88b2f1803449789c36f14fb4d3c1eb.pt",
        "mel_len": 652
      },
      ...],
      "id10328": [
      {
        "feature_path": "uttr-b3d49b13bb36497186c923cd8f1b811e.pt",
        "mel_len": 989
      },
      ...],
      ...
      }
    """

    # 总共有几个speaker
    self.speaker_num = len(metadata.keys())
    self.data = []
    # 遍历所有说话的人以及他们的音频
    for speaker in metadata.keys():
      for voice in metadata[speaker]:
        self.data.append([f"Dataset/{voice["feature_path"]}", self.speaker2id[speaker]]) #存储为[音频路径, speaker id]的形式

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    feature_path, speaker = self.data[index]
    # 加载处理好的梅尔频谱图。
    mel = torch.load(feature_path)

    # 统一语音片段的长度
    if len(mel) > self.segment_len:
      # 随机选取片段的起始点
      start = np.random.randint(0, len(mel) - self.segment_len + 1)
      # 获取分割后的片段
      mel = mel[start : start+self.segment_len].detach().clone().to(torch.float32)
    else:
        # 如果语音片段长度不足，则直接拿回原片段
        mel = mel.detach().clone().to(torch.float32)
        
    # 将speaker id 转换为长整型以用于交叉熵运算
    speaker = torch.tensor(speaker, dtype=torch.long)
    return mel, speaker

  def get_speaker_number(self):
    return self.speaker_num

定义模型

In [3]:
class Classifier(nn.Module):
    def __init__(self, d_model=80, n_speakers=600, dropout=0.1):
        super().__init__()

        #预处理层: 将输入的语音特征 (维度通常为 40) 映射到 d_model 的维度
        self.prenet = nn.Sequential(
            nn.Linear(40, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Self-Attention 核心机制 (Transformer Encoder)
        # batch_first=True 表示输入数据的形状是 (batch, seq, feature)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            dim_feedforward=256,
            nhead=4,         # 多头注意力的头数
            dropout=dropout,
            batch_first=True
        )

        # 堆叠多层 (num_layers) 的 Transformer 块
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # 输出分类层
        # 输入维度为 d_model，输出为说话人的总数 (600)
        self.prediction_layer = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_speakers),
        )

    def forward(self, mels):
        """
        输入:
            mels: (batch size, length, 40) 声音信号的特征图
        返回:
            out: (batch size, n_speakers) 预测结果
        """
        # out: (batch size, length, d_model)
        out = self.prenet(mels)

        # 进行 Self-Attention 计算
        # out: (batch size, length, d_model)
        out = self.encoder(out)

        # 4. 池化层 (Pooling Layer)
        # 因为 Self-Attention 不改变序列长度，所以最后需要在 length 维度上进行汇聚
        out = out.mean(dim=1)  # out: (batch size, d_model)

        # 5. 全连接层分类预测
        # out: (batch size, n_speakers)
        out = self.prediction_layer(out)

        return out

划分训练集和验证集并加载

In [4]:
def collate_batch(batch):
  mel, speaker = zip(*batch)
  mel = pad_sequence(mel, batch_first=True, padding_value=-20)# 填充 log 10^(-20) 的值，表示没有声音的语音片段
  # mel: (batch size, length, 40)
  return mel, torch.tensor(speaker, dtype=torch.long)

dataset = SpeakerDataset()
total_size = len(dataset)
train_size = int(0.8 * total_size)
validation_size = total_size - train_size
generator = torch.Generator().manual_seed(42)
train_set, validation_set = random_split(dataset, [train_size, validation_size], generator=generator)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_batch)
validation_loader = DataLoader(validation_set, batch_size=32, shuffle=False, collate_fn=collate_batch)

开始训练

In [5]:
model = Classifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 100
best_accuracy = 0
for epoch in range(epochs):
	model.train()
	total_loss = 0
	for mels, speakers in train_loader:
		mels, speakers = mels.to(device), speakers.to(device)
		optimizer.zero_grad()
		outputs = model(mels)
		loss = criterion(outputs, speakers)
		loss.backward()
		optimizer.step()
		total_loss += loss.item()
	avg_loss = total_loss / len(train_loader)
	print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss}")

	model.eval()
	correct, total = 0, 0
	with torch.no_grad():
		for mels, speakers in validation_loader:
			mels, speakers = mels.to(device), speakers.to(device)
			outputs = model(mels)
			_, predicted = torch.max(outputs.data, 1)
			total += speakers.size(0)
			correct += (predicted == speakers).sum().item()
	accuracy = correct / total
	print(f"Validation Accuracy: {accuracy}")
	if accuracy > best_accuracy:
		best_accuracy = accuracy
		torch.save(model.state_dict(), "best_model.pth")
		print(f"Best model saved with accuracy: {best_accuracy}")


Epoch [1/100], Loss: 4.699213431864839


C:\Users\A\AppData\Local\Temp\ipykernel_21952\4107737668.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mel = torch.tensor(mel, dtype=torch.float32)


Validation Accuracy: 0.2078052995391705
Best model saved with accuracy: 0.2078052995391705
Epoch [2/100], Loss: 3.397106435716427
Validation Accuracy: 0.33942972350230416
Best model saved with accuracy: 0.33942972350230416
Epoch [3/100], Loss: 2.827966419507831
Validation Accuracy: 0.4097782258064516
Best model saved with accuracy: 0.4097782258064516
Epoch [4/100], Loss: 2.4824628498971735
Validation Accuracy: 0.4444124423963134
Best model saved with accuracy: 0.4444124423963134
Epoch [5/100], Loss: 2.239527822227522
Validation Accuracy: 0.49330357142857145
Best model saved with accuracy: 0.49330357142857145
Epoch [6/100], Loss: 2.0554952210132975
Validation Accuracy: 0.5359302995391705
Best model saved with accuracy: 0.5359302995391705
Epoch [7/100], Loss: 1.9161438686163743
Validation Accuracy: 0.5524193548387096
Best model saved with accuracy: 0.5524193548387096
Epoch [8/100], Loss: 1.7938278455132712
Validation Accuracy: 0.5614199308755761
Best model saved with accuracy: 0.56141993

构造推理时的数据集

In [6]:
class InferenceDataset(Dataset):
  def __init__(self):
    
    testdata = json.load(open("Dataset/testdata.json"))
    self.data = testdata["utterances"]
    """
  	数据的形式：
    [{"feature_path": "uttr-b73206c2bc3d42bf9c77ad4565d4ff15.pt", "mel_len": 2112},...]
  	"""

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    voice = self.data[index]
    feature_path = voice["feature_path"]
    mel = torch.load(f"Dataset/{feature_path}")

    return feature_path, mel

加载推理数据

In [7]:
def inference_collate_batch(batch):
  """Collate a batch of data."""
  features_paths, mels = zip(*batch)

  return features_paths, torch.stack(mels)

inference_set = InferenceDataset()
inference_loader = DataLoader(inference_set, batch_size=1, shuffle=False, collate_fn=inference_collate_batch)#推理时每次只取一条语音

推理并输出

In [8]:
import csv
model.load_state_dict(torch.load('./best_model.pth'))
model.eval()
results = []
with torch.no_grad():
    for feature_paths, mels in inference_loader:
        mels = mels.to(device)
        outputs = model(mels)
        _, predicted = torch.max(outputs.data, 1)
        for path, pred in zip(feature_paths, predicted):
            results.append([path, pred.item()])

with open('prediction.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['feature_path', 'speaker id'])
    writer.writerows(results)
print("推理完成，结果已保存到 prediction.csv")

推理完成，结果已保存到 prediction.csv
